# NIDS: Sequential Pattern Mining + Multi-Modal Data Fusion on CIC-IDS2017

Condensed 8-cell implementation of the Flow-Aware Temporal Pattern Mining
methodology (session reconstruction, behavioral event encoding, FP-Growth +
PrefixSpan sequential pattern mining, a temporal attack-state graph, and
**early (feature-level) multi-modal fusion** across three modalities:

1. **Statistical modality** -- session-aggregated CICFlowMeter features.
2. **Sequential-pattern modality** -- behavioral-token bag-of-tokens +
   the FP-Growth/PrefixSpan pattern-match score `SP_t`.
3. **Graph-temporal modality** -- attack-state-graph consistency `TC_t`,
   structural path weight `G_w`, and normalised inter-event timing.

These are concatenated into one fused feature vector per **session** (not
per flow), which Cell 6 trains classifiers on.

**Methodology corrections applied** (from the reviewer pass on the original
draft): chronological session-aware split (Train=Mon+Tue / Val=Wed /
Test=Thu+Fri, never random-stratified); SMOTE-KNN restricted to RF/XGBoost
for classes with >=50 training samples, never touching the sequential/graph
modalities; class weighting as the primary imbalance strategy; Heartbleed
and Infiltration reported per-class, never merged into a rare-class bucket.

This notebook reuses the tested `nids` package in `../src/nids` (built and
smoke-tested earlier in this project) rather than re-deriving ~2000 lines of
pipeline logic inline -- each cell below is a thin, readable orchestration
layer with its own error handling, per your request.

**Dataset path:** `D:\IDSPROJECT2026\CIC-IDS2017` (edit `DATA_DIR` in Cell 1
if yours differs).


In [ ]:
# ============================================================
# CELL 1 -- Imports and setup
# ============================================================
import sys
import logging
import warnings
from pathlib import Path

try:
    # Make the project's `nids` package importable from this notebook.
    PROJECT_ROOT = Path("..").resolve()
    SRC_DIR = PROJECT_ROOT / "src"
    if str(SRC_DIR) not in sys.path:
        sys.path.insert(0, str(SRC_DIR))

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

    from nids import (
        config, data_loading, preprocessing, features, sessions, events,
        pattern_mining, attack_graph, risk, models, metrics as nids_metrics,
        pipeline,
    )

    warnings.filterwarnings("ignore", category=FutureWarning)
    logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(name)s:%(message)s")
    logger = logging.getLogger("nids_notebook")

    np.random.seed(config.RANDOM_SEED)
    pd.set_option("display.max_columns", 60)
    pd.set_option("display.width", 160)
    sns.set_theme(style="whitegrid")

    # --- EDIT THIS if your dataset lives somewhere else ---------------------
    DATA_DIR = Path(r"D:\IDSPROJECT2026\CIC-IDS2017")
    # -------------------------------------------------------------------------

    print("Setup OK.")
    print("nids package loaded from:", SRC_DIR)
    print("DATA_DIR:", DATA_DIR)
    print("Classes (K=%d):" % config.NUM_CLASSES, config.CLASSES)

except ImportError as e:
    raise ImportError(
        "A required package is missing. From the project root run:\n"
        "    pip install -r requirements.txt\n"
        f"Original error: {e}"
    )
except Exception as e:
    print(f"[Cell 1 ERROR] Setup failed: {type(e).__name__}: {e}")
    raise


In [ ]:
# ============================================================
# CELL 2 -- Load dataset (Stage 1: Data Acquisition & Partition)
# ============================================================
# Auto-discovers the 7-8 daily CIC-IDS2017 CSVs under DATA_DIR by weekday
# name, normalizes column names/labels, tags provenance (day/day_order/
# source_file -- metadata only, NEVER used as a model feature), and sorts
# chronologically. All 78 features and all 15 class labels are retained --
# Heartbleed and Infiltration are never merged into a rare-class bucket.
try:
    raw_df = data_loading.load_raw_data(DATA_DIR)

    print(f"Loaded {len(raw_df):,} flow records, {raw_df.shape[1]} columns.")
    print("\nFlows per day:")
    print(raw_df["day"].value_counts().reindex(config.DAY_NAMES))
    print("\nClass distribution (top 15):")
    print(raw_df[config.LABEL_COLUMN].value_counts().head(15))

except FileNotFoundError as e:
    print(f"[Cell 2 ERROR] Could not find the dataset: {e}")
    print("Check that DATA_DIR (set in Cell 1) points at the folder containing")
    print("the CIC-IDS2017 daily CSV files (Monday-WorkingHours.pcap_ISCX.csv, etc).")
    raise
except Exception as e:
    print(f"[Cell 2 ERROR] Failed to load dataset: {type(e).__name__}: {e}")
    raise


In [ ]:
# ============================================================
# CELL 3 -- Data preprocessing (Stage 2: Cleaning, Chronological Split,
#           Imbalance Handling)
# ============================================================
try:
    # Inf -> NaN -> drop (CIC-IDS2017's known ~1.8% corrupt rows). Only
    # numeric feature columns are inspected; IDs/labels/metadata untouched.
    feature_cols_raw = features.candidate_feature_columns(raw_df)
    clean_df = data_loading.clean_data(raw_df, numeric_cols=feature_cols_raw)

    # CHRONOLOGICAL, session-aware split (Review-corrected: never random-
    # stratified). Train=Mon+Tue, Val=Wed, Test=Thu+Fri. A session can never
    # span a day boundary (Stage 4), so this split is leakage-free by
    # construction -- no session straddles train/val/test.
    train_df, val_df, test_df = data_loading.split_partitions(clean_df)
    print(f"train/val/test flow counts: {len(train_df):,} / {len(val_df):,} / {len(test_df):,}")

    # IMPORTANT CAVEAT: under this split, Heartbleed (Wednesday-only in the
    # real dataset) and Infiltration (Thursday-only) can have ZERO training
    # examples -- this is expected, not a bug. Report it in your Threats to
    # Validity section rather than silently merging/dropping those classes.
    zero_shot = pipeline.check_zero_shot_classes(train_df[config.LABEL_COLUMN])

    # Fit scaling (median-impute -> variance-filter -> Min-Max scale) on
    # TRAINING data only; transform val/test without re-fitting.
    preprocessor = preprocessing.TrainOnlyPreprocessor(feature_cols_raw).fit(train_df)
    X_train = preprocessor.transform(train_df)
    X_val = preprocessor.transform(val_df)
    X_test = preprocessor.transform(test_df)
    print(f"Scaled feature matrix shapes: train={X_train.shape}, val={X_val.shape}, test={X_test.shape}")

    # Branch A (PRIMARY): class weights W_c = N_train / (K * N_c), applied
    # to every model including the LSTM's loss.
    class_weights = preprocessing.compute_class_weights(train_df[config.LABEL_COLUMN])
    print("\nClass weights (non-zero only):")
    print(pd.Series({k: v for k, v in class_weights.items() if v > 0}).sort_values(ascending=False))

except Exception as e:
    print(f"[Cell 3 ERROR] Preprocessing failed: {type(e).__name__}: {e}")
    raise


In [ ]:
# ============================================================
# CELL 4 -- Feature Engineering & Sequential Pattern Mining
#           (Stages 3-5, 8: feature groups, session reconstruction,
#            behavioral event encoding, FP-Growth + PrefixSpan)
# ============================================================
try:
    # Stage 3: partition the 78 features into 4 domain-knowledge groups
    # (A: statistical->RF, B: protocol->shared, C: temporal->LSTM,
    # D: TCP flags->XGBoost), MI-ranked on the TRAINING split only.
    feature_sets = features.build_model_feature_sets(X_train, train_df[config.LABEL_COLUMN])
    print("Feature groups:", {k: len(v) for k, v in feature_sets["groups"].items()})

    # Stage 4: bidirectional session reconstruction (symmetric 5-tuple key,
    # tau=60s idle timeout) -- run independently per partition so no session
    # spans a day/partition boundary.
    train_df = sessions.reconstruct_sessions(train_df)
    val_df = sessions.reconstruct_sessions(val_df)
    test_df = sessions.reconstruct_sessions(test_df)
    print(f"\nSessions -- train: {train_df['session_id'].nunique():,}, "
          f"val: {val_df['session_id'].nunique():,}, test: {test_df['session_id'].nunique():,}")

    # Stage 5: rule-based behavioral event encoding (10-token vocabulary,
    # derived from flow attributes only -- never the ground-truth label).
    train_df["token"] = events.encode_events(train_df)
    val_df["token"] = events.encode_events(val_df)
    test_df["token"] = events.encode_events(test_df)
    print("\nBehavioral token distribution (train):")
    print(train_df["token"].value_counts())

    # Stage 8: FP-Growth (unordered co-occurrence) + PrefixSpan (ordered,
    # gap-constrained) sequential pattern mining -- TRAINING sessions only,
    # on ORIGINAL tokens only (never SMOTE synthetic data).
    train_seqs = events.build_session_sequences(train_df)
    train_tokens_only = {sid: [t for t, _ in seq] for sid, seq in train_seqs.items()}
    fp_patterns = pattern_mining.mine_fp_growth(train_tokens_only)
    seq_patterns = pattern_mining.mine_prefixspan(train_seqs)
    print(f"\nMined {len(fp_patterns)} FP-Growth itemsets, {len(seq_patterns)} PrefixSpan sequences.")
    if not fp_patterns.empty:
        print(fp_patterns.head(5))

except Exception as e:
    print(f"[Cell 4 ERROR] Feature engineering / pattern mining failed: {type(e).__name__}: {e}")
    raise


In [ ]:
# ============================================================
# CELL 5 -- Multi-Modal Data Fusion (Stages 6/9-11 inputs, fused early)
# ============================================================
# Builds ONE fused feature vector per SESSION by concatenating three
# modalities -- this is feature-level ("early") fusion, distinct from the
# probability-level ("late") fusion used elsewhere in the full architecture:
#
#   1. STATISTICAL modality : mean-pooled scaled CICFlowMeter features
#                              across the session's flows.
#   2. SEQUENTIAL modality   : normalized bag-of-behavioral-tokens (10-dim)
#                              + the FP-Growth/PrefixSpan pattern score SP_t.
#   3. GRAPH-TEMPORAL modality: attack-state-graph consistency TC_t,
#                              structural path weight G_w, and normalized
#                              inter-event timing (1/dt_norm).
#
# The attack-state graph itself is built from TRAINING sessions only, and
# lambda (TC_t's temporal decay constant) is selected on the VALIDATION
# split by maximising ROC-AUC of TC_t as an attack/benign separator
# (Review §17 -- must be empirically validated, not assumed).
try:
    def session_statistical_modality(df, X_scaled, feature_cols):
        """Modality 1: mean-pool each session's scaled flow features."""
        block = X_scaled.loc[df.index, feature_cols].copy()
        block["session_id"] = df["session_id"].values
        return block.groupby("session_id")[feature_cols].mean()

    def session_sequential_modality(df, seqs, fp_patterns, seq_patterns):
        """Modality 2: bag-of-tokens (normalized counts) + SP_t."""
        bow = (
            pd.crosstab(df["session_id"], df["token"])
            .reindex(columns=config.TOKENS, fill_value=0)
        )
        bow = bow.div(bow.sum(axis=1).clip(lower=1), axis=0)  # normalize per session
        bow.columns = [f"tok_{c}" for c in bow.columns]
        sp_t = pd.Series(
            {sid: pattern_mining.compute_sp_t(seqs[sid], fp_patterns, seq_patterns) for sid in seqs},
            name="SP_t",
        )
        return bow.join(sp_t)

    def session_graph_modality(seqs, graph, lam, mean_train_gap):
        """Modality 3: TC_t, G_w, 1/dt_norm from the temporal attack-state graph."""
        tc_t = pd.Series({sid: attack_graph.compute_tc_t(seqs[sid], graph, lam) for sid in seqs}, name="TC_t")
        g_w = pd.Series({sid: attack_graph.compute_g_w(seqs[sid], graph) for sid in seqs}, name="G_w")
        gap = pd.Series({sid: risk.compute_session_mean_gap(seqs[sid]) for sid in seqs})
        inv_dt = pd.Series(risk.normalize_delta_t(gap.values, mean_train_gap), index=gap.index, name="inv_dt_norm")
        return pd.concat([tc_t, g_w, inv_dt], axis=1)

    def build_fused_dataset(df, X_scaled, feature_cols, fp_patterns, seq_patterns, graph, lam, mean_train_gap):
        seqs = events.build_session_sequences(df)
        stat = session_statistical_modality(df, X_scaled, feature_cols)
        seq_mod = session_sequential_modality(df, seqs, fp_patterns, seq_patterns)
        graph_mod = session_graph_modality(seqs, graph, lam, mean_train_gap)
        summ = sessions.session_summary(df).set_index("session_id")

        fused = stat.join(seq_mod, how="left").join(graph_mod, how="left")
        fused = fused.join(summ[["label", "day", "partition", "n_flows", "start_time", "end_time"]])
        fused["is_attack"] = (fused["label"] != "BENIGN").astype(int)
        return fused.fillna(0.0) if fused.isna().any().any() else fused, seqs

    all_feature_cols = sorted(str(c) for c in set(feature_sets["rf"]) | set(feature_sets["xgb"]) | set(feature_sets["lstm_context"]))

    # Build the attack-state graph from TRAINING sessions only.
    graph = attack_graph.AttackStateGraph().build_from_training(train_seqs)
    mean_train_gap = risk.compute_mean_interevent_time(train_seqs)

    # Select lambda on the VALIDATION split (needs a first-pass TC_t, so we
    # bootstrap with a mid-range candidate then re-select).
    lam0 = config.TC_LAMBDA_CANDIDATES[len(config.TC_LAMBDA_CANDIDATES) // 2]
    val_seqs_tmp = events.build_session_sequences(val_df)
    val_is_attack = {sid: int(sessions.session_summary(val_df).set_index("session_id").loc[sid, "label"] != "BENIGN")
                      for sid in val_seqs_tmp}
    lam, auc = attack_graph.select_lambda(val_seqs_tmp, graph, val_is_attack)
    print(f"Selected TC_t lambda={lam} (validation ROC-AUC={auc:.4f})")

    fused_train, seqs_train = build_fused_dataset(train_df, X_train, all_feature_cols, fp_patterns, seq_patterns, graph, lam, mean_train_gap)
    fused_val, seqs_val = build_fused_dataset(val_df, X_val, all_feature_cols, fp_patterns, seq_patterns, graph, lam, mean_train_gap)
    fused_test, seqs_test = build_fused_dataset(test_df, X_test, all_feature_cols, fp_patterns, seq_patterns, graph, lam, mean_train_gap)

    fusion_feature_cols = all_feature_cols + [f"tok_{t}" for t in config.TOKENS] + ["SP_t", "TC_t", "G_w", "inv_dt_norm"]
    print(f"\nFused feature vector dimensionality: {len(fusion_feature_cols)} "
          f"({len(all_feature_cols)} statistical + {len(config.TOKENS)} sequential + 1 SP_t + 3 graph-temporal)")
    print(f"Fused session datasets: train={len(fused_train)}, val={len(fused_val)}, test={len(fused_test)}")
    fused_train[fusion_feature_cols + ["label"]].head()

except Exception as e:
    print(f"[Cell 5 ERROR] Multi-modal fusion failed: {type(e).__name__}: {e}")
    raise


In [ ]:
# ============================================================
# CELL 6 -- Model training (Stage 6: trained on the FUSED multi-modal
#           feature matrix, plus a sequence-native BiLSTM for comparison)
# ============================================================
try:
    y_train_sess = fused_train["label"]
    y_val_sess = fused_val["label"]
    y_test_sess = fused_test["label"]

    # --- Random Forest & XGBoost on the fused multi-modal vector ----------
    # Branch A (PRIMARY, Review §8): class weighting. Every session-level
    # row already blends statistical + sequential + graph-temporal evidence.
    rf_fused = models.train_rf(fused_train[fusion_feature_cols], y_train_sess, class_weights=class_weights)
    xgb_fused = models.train_xgb(fused_train[fusion_feature_cols], y_train_sess, class_weights=class_weights)

    proba_rf_test = models.predict_proba_rf(rf_fused, fused_test[fusion_feature_cols])
    proba_xgb_test = models.predict_proba_xgb(xgb_fused, fused_test[fusion_feature_cols])

    # --- Optional experimental comparison: Branch B (SMOTE-KNN) -----------
    # Restricted to RF/XGBoost, and only classes with >= config.SMOTE_MIN_CLASS_COUNT
    # training samples (Review §4/§8 correction) -- run for the A5 ablation.
    RUN_SMOTE_BRANCH_B = False
    if RUN_SMOTE_BRANCH_B:
        X_smote, y_smote = preprocessing.smote_knn_resample(fused_train[fusion_feature_cols], y_train_sess)
        rf_fused_b = models.train_rf(X_smote, y_smote, class_weights=class_weights)
        print("Branch B (SMOTE-KNN) RF trained on", len(X_smote), "rows (was", len(fused_train), ")")

    # --- BiLSTM on raw behavioral token sequences (sequence-native model) -
    # Never touched by SMOTE (Review §8/§11) -- trained on original sessions.
    ordered_train_sids = list(seqs_train.keys())
    seq_idx_train = [events.sequence_to_indices(seqs_train[s]) for s in ordered_train_sids]
    seq_labels_train = [y_train_sess.loc[s] for s in ordered_train_sids]

    lstm = models.BiLSTMClassifier()
    lstm.fit(seq_idx_train, seq_labels_train, class_weights, epochs=config.LSTM_EPOCHS)

    ordered_test_sids = list(seqs_test.keys())
    seq_idx_test = [events.sequence_to_indices(seqs_test[s]) for s in ordered_test_sids]
    proba_lstm_test = lstm.predict_proba(seq_idx_test)
    proba_lstm_test_df = pd.DataFrame(proba_lstm_test, columns=config.CLASSES, index=ordered_test_sids).reindex(fused_test.index)

    # --- Simple late-fusion ensemble: average the fused-classifier and the
    # sequence-native LSTM's predictions (demonstrates combining early- and
    # late-fusion evidence, matching the full architecture's Stage 7 spirit).
    proba_ensemble_test = (proba_rf_test + proba_xgb_test + proba_lstm_test_df.values) / 3.0

    predictions = {
        "RandomForest_fused": [config.CLASSES[i] for i in proba_rf_test.argmax(1)],
        "XGBoost_fused": [config.CLASSES[i] for i in proba_xgb_test.argmax(1)],
        "BiLSTM_sequential": [config.CLASSES[i] for i in proba_lstm_test.argmax(1)],
        "Ensemble_avg": [config.CLASSES[i] for i in proba_ensemble_test.argmax(1)],
    }
    print("Models trained:", list(predictions.keys()))

except Exception as e:
    print(f"[Cell 6 ERROR] Model training failed: {type(e).__name__}: {e}")
    raise


In [ ]:
# ============================================================
# CELL 7 -- Evaluation and results (Stage 24: Macro-F1, per-class F1, FPR)
# ============================================================
# Accuracy is reported last / with caution -- BENIGN dominance makes it an
# uninformative headline number (Review §24). Heartbleed/Infiltration are
# always shown per-class, never merged away (Review §4).
try:
    summary_rows = []
    per_class_reports = {}
    for model_name, y_pred in predictions.items():
        macro_f1 = nids_metrics.macro_f1(y_test_sess, y_pred)
        fpr = nids_metrics.false_positive_rate(y_test_sess, y_pred)
        acc = float(np.mean(np.array(y_pred) == y_test_sess.values))
        summary_rows.append({"model": model_name, "macro_f1": macro_f1, "fpr": fpr, "accuracy": acc})
        per_class_reports[model_name] = nids_metrics.per_class_report(y_test_sess, y_pred)

    summary_df = pd.DataFrame(summary_rows).sort_values("macro_f1", ascending=False)
    print("=== Model comparison on TEST sessions ===")
    print(summary_df.to_string(index=False))

    best_model = summary_df.iloc[0]["model"]
    print(f"\n=== Per-class report for best model ({best_model}) ===")
    best_report = per_class_reports[best_model]
    print(best_report.to_string(index=False))

    # 95% bootstrap confidence interval on the best model's macro-F1.
    pt, lo, hi = nids_metrics.bootstrap_ci(
        y_test_sess.values, np.array(predictions[best_model]), nids_metrics.macro_f1, n_boot=1000
    )
    print(f"\nMacro-F1 95% bootstrap CI ({best_model}): {pt:.4f} [{lo:.4f}, {hi:.4f}]")

    # Statistical significance: fused ensemble vs the single best fused
    # tabular model (Review §21/§26-G).
    stat, p = nids_metrics.mcnemar_test(y_test_sess.values, predictions["Ensemble_avg"], predictions["RandomForest_fused"])
    print(f"McNemar's test (Ensemble_avg vs RandomForest_fused): stat={stat:.3f}, p={p:.4f}")

    confusion_best = nids_metrics.confusion(y_test_sess, predictions[best_model])

except Exception as e:
    print(f"[Cell 7 ERROR] Evaluation failed: {type(e).__name__}: {e}")
    raise


In [ ]:
# ============================================================
# CELL 8 -- Visualization of results
# ============================================================
try:
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))

    # (1) Model comparison: macro-F1 and FPR
    ax = axes[0, 0]
    x = np.arange(len(summary_df))
    width = 0.35
    ax.bar(x - width / 2, summary_df["macro_f1"], width, label="Macro-F1")
    ax.bar(x + width / 2, summary_df["fpr"], width, label="FPR")
    ax.set_xticks(x)
    ax.set_xticklabels(summary_df["model"], rotation=20, ha="right")
    ax.set_title("Model comparison on test sessions")
    ax.legend()

    # (2) Per-class F1 for the best model (Heartbleed/Infiltration included)
    ax = axes[0, 1]
    per_class_plot = best_report[best_report["class"] != "MACRO_AVG"].sort_values("f1")
    colors = ["crimson" if c in ("Heartbleed", "Infiltration") else "steelblue" for c in per_class_plot["class"]]
    ax.barh(per_class_plot["class"], per_class_plot["f1"], color=colors)
    ax.set_title(f"Per-class F1 ({best_model}) -- red = zero-shot-risk classes")
    ax.set_xlabel("F1 score")

    # (3) Confusion matrix heatmap (best model)
    ax = axes[1, 0]
    sns.heatmap(confusion_best, annot=False, cmap="Blues", ax=ax, cbar_kws={"label": "count"})
    ax.set_title(f"Confusion matrix ({best_model})")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.tick_params(axis="x", rotation=90)
    ax.tick_params(axis="y", rotation=0)

    # (4) Behavioral token distribution feeding the sequential modality
    ax = axes[1, 1]
    train_df["token"].value_counts().reindex(config.TOKENS, fill_value=0).plot(kind="bar", ax=ax, color="darkorange")
    ax.set_title("Behavioral token distribution (train)")
    ax.set_ylabel("count"); ax.tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()

    # Modality contribution snapshot: fused-classifier vs sequence-only LSTM
    # vs the 3-way ensemble, isolating how much the sequential/graph
    # modalities add over the statistical modality alone.
    fig2, ax2 = plt.subplots(figsize=(7, 4))
    ax2.bar(summary_df["model"], summary_df["macro_f1"], color="teal")
    ax2.set_ylabel("Macro-F1"); ax2.set_title("Macro-F1 by modality combination")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"[Cell 8 ERROR] Visualization failed: {type(e).__name__}: {e}")
    raise
